# Course in Python and AI at Lexicon
## Projekt work dataanalys

In [83]:
import matplotlib.pyplot as plt
import pandas as pd



In [84]:
flights_original = pd.read_csv('data/flights.csv')
flights = flights_original.copy()
airports_original = pd.read_csv('data/airports.csv')
airports = airports_original.copy()

print(flights.shape)
print(flights.info())
print(flights.describe())
print()
print('-----------------------------------------')
print()
print(airports.shape)
print(airports.info())
print(airports.describe())


(20000, 5)
<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   date         20000 non-null  str  
 1   delay        20000 non-null  int64
 2   distance     20000 non-null  int64
 3   origin       20000 non-null  str  
 4   destination  20000 non-null  str  
dtypes: int64(2), str(3)
memory usage: 781.4 KB
None
              delay      distance
count  20000.000000  20000.000000
mean       7.703900    723.846700
std       31.318537    562.705585
min      -59.000000     30.000000
25%       -8.000000    309.000000
50%        0.000000    562.000000
75%       13.000000    978.000000
max      522.000000   4475.000000

-----------------------------------------

(3376, 7)
<class 'pandas.DataFrame'>
RangeIndex: 3376 entries, 0 to 3375
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   iata       3376 non

## The Data

The data comes from the Vega datasets on GitHub (https://vega.github.io/vega-datasets/). It seems to be the datasets 'flights_20k' and 'airports' we have been given here. The datasets are example data for the diagram tool Vega. The Vega project made this file as a demo for its charting tools, not to study delays. It is a sample of a larger file, and I do not know how it was chosen.

Flight delay statistics (20,000 rows) from U.S. Bureau of Transportation Statistics. Collected under regulatory reporting requirements, which mandate that qualifying airlines report on-time performance data to BTS. Date follows the format %Y-%m-%d %H:%M:%S, delay seems to be given in minutes. Licence for the data is unclear.

Airports in the United States and its territories, including major commercial, regional, and municipal airports. Contains information about each airport's location (latitude/longitude coordinates), identification codes, name, city, state, and country. While the exact generation source of this file is unknown, this data is consistent with files provided on a monthly frequency by the FAA's National Airspace System Resource. License: https://www.usa.gov/government-works .

### Sources

1. Vega datasets, metadata overview: https://vega.github.io/vega-datasets/metadata/
2. Vega datasets, airports: https://vega.github.io/vega-datasets/datasets/airports/

In [85]:
flights["date"].head(10)

0    2001-01-01 00:47:00
1    2001-01-01 01:10:00
2    2001-01-01 01:24:00
3    2001-01-01 01:39:00
4    2001-01-01 06:02:00
5    2001-01-01 06:05:00
6    2001-01-01 06:17:00
7    2001-01-01 06:22:00
8    2001-01-01 06:30:00
9    2001-01-01 06:35:00
Name: date, dtype: str

In [86]:
flights["date"] = pd.to_datetime(flights["date"], format="%Y-%m-%d %H:%M:%S", errors="coerce")

print(flights["date"].isna().sum())
print(flights["date"].min(), flights["date"].max())
print(flights["date"].dtype)

0
2001-01-01 00:47:00 2001-03-31 22:27:00
datetime64[us]


In [87]:
# Looking for the 12 missing Airport codes and cities

missing = airports[airports["city"].isna()]
print(missing[["iata", "name", "city", "state", "country"]])

     iata                          name city state  \
1136  CLD    MC Clellan-Palomar Airport  NaN   NaN   
1715  HHH                   Hilton Head  NaN   NaN   
2251  MIB                     Minot AFB  NaN   NaN   
2312  MQT      Marquette County Airport  NaN   NaN   
2752  RCA                 Ellsworth AFB  NaN   NaN   
2759  RDR               Grand Forks AFB  NaN   NaN   
2794  ROP                   Prachinburi  NaN   NaN   
2795  ROR              Babelthoup/Koror  NaN   NaN   
2900  SCE               University Park  NaN   NaN   
2964  SKA                 Fairchild AFB  NaN   NaN   
3001  SPN  Tinian International Airport  NaN   NaN   
3355  YAP             Yap International  NaN   NaN   

                             country  
1136                             USA  
1715                             USA  
2251                             USA  
2312                             USA  
2752                             USA  
2759                             USA  
2794                    

In [88]:
codes = ["CLD", "HHH", "MIB", "MQT", "RCA", "RDR", "SCE", "SKA", "ROP", "ROR", "SPN", "YAP"]

ourairports = pd.read_csv("data/ourairports_airports.csv", keep_default_na=False)

fill = ourairports[ourairports["iata_code"].isin(codes)].copy()
print(fill[["iata_code", "name", "municipality", "iso_region", "iso_country"]])

      iata_code                                  name        municipality  \
39277       CLD             McClellan-Palomar Airport            Carlsbad   
40368       HHH                   Hilton Head Airport  Hilton Head Island   
40854       MIB                  Minot Air Force Base               Minot   
42599       RCA              Ellsworth Air Force Base          Rapid City   
42611       RDR            Grand Forks Air Force Base         Grand Forks   
42747       MQT     Marquette Sawyer Regional Airport               Gwinn   
42807       SKA              Fairchild Air Force Base             Spokane   
43088       SCE        State College Regional Airport       State College   
53284       ROP            Rota International Airport         Rota Island   
53285       SPN          Saipan International Airport    I Fadang, Saipan   
54853       ROR  Roman Tmetuchl International Airport   Babelthuap Island   
54855       YAP             Yap International Airport          Yap Island   

In [89]:
codes = ["CLD", "HHH", "MIB", "MQT", "RCA", "RDR", "SCE", "SKA", "ROP", "ROR", "SPN", "YAP"]

ourairports = pd.read_csv("data/ourairports_airports.csv", keep_default_na=False)

fill = ourairports[ourairports["iata_code"].isin(codes)].copy()
fill["state"] = fill["iso_region"].where(fill["iso_country"] == "US").str.replace("US-", "", regex=False)
fill = fill.set_index("iata_code")

print(fill[["name", "municipality", "iso_country", "state"]])

                                           name        municipality  \
iata_code                                                             
CLD                   McClellan-Palomar Airport            Carlsbad   
HHH                         Hilton Head Airport  Hilton Head Island   
MIB                        Minot Air Force Base               Minot   
RCA                    Ellsworth Air Force Base          Rapid City   
RDR                  Grand Forks Air Force Base         Grand Forks   
MQT           Marquette Sawyer Regional Airport               Gwinn   
SKA                    Fairchild Air Force Base             Spokane   
SCE              State College Regional Airport       State College   
ROP                  Rota International Airport         Rota Island   
SPN                Saipan International Airport    I Fadang, Saipan   
ROR        Roman Tmetuchl International Airport   Babelthuap Island   
YAP                   Yap International Airport          Yap Island   

     

In [90]:
print("Before:", airports["city"].isna().sum(), airports["state"].isna().sum())

airports["city"] = airports["city"].fillna(airports["iata"].map(fill["municipality"]))
airports["state"] = airports["state"].fillna(airports["iata"].map(fill["state"]))

print("After:", airports["city"].isna().sum(), airports["state"].isna().sum())
print(airports[airports["iata"] == "MQT"])

Before: 12 12
After: 0 4
     iata                      name   city state country   latitude  longitude
2312  MQT  Marquette County Airport  Gwinn    MI     USA  46.353639 -87.395361


In [91]:
# Chcking for duplicates before merge
print("Duplicate iata in airports:", airports["iata"].duplicated().sum())
print("Duplicate rows in flights:", flights.duplicated().sum())

# An empty set means every code in flights exists in airports.
print("Origin not in airports:", set(flights["origin"]) - set(airports["iata"])) # 
print("Destination not in airports:", set(flights["destination"]) - set(airports["iata"]))

Duplicate iata in airports: 0
Duplicate rows in flights: 0
Origin not in airports: set()
Destination not in airports: set()


In [92]:
airports_small = airports[["iata", "city", "state", "latitude", "longitude"]]

departure_info = airports_small.add_prefix("departure_")
arrival_info = airports_small.add_prefix("arrival_")

flights_merged = flights.merge(
    departure_info, left_on="origin", right_on="departure_iata",
    how="left", validate="many_to_one"
).drop(columns="departure_iata")

flights_merged = flights_merged.merge(
    arrival_info, left_on="destination", right_on="arrival_iata",
    how="left", validate="many_to_one"
).drop(columns="arrival_iata")

print("Rows before:", len(flights))
print("Rows after:", len(flights_merged))
print(flights_merged.isna().sum())

Rows before: 20000
Rows after: 20000
date                   0
delay                  0
distance               0
origin                 0
destination            0
departure_city         0
departure_state        0
departure_latitude     0
departure_longitude    0
arrival_city           0
arrival_state          0
arrival_latitude       0
arrival_longitude      0
dtype: int64
